# Global COVID-19 Policy and Vaccination Impact Analysis

This notebook analyses the impact of government policies and vaccination campaigns on COVID-19 case counts and mortality during the pandemic.

### Data Sources
1. **JHU CSSE** — Johns Hopkins University (case and death counts)
2. **OWID** — Our World in Data (vaccination data)
3. **OxCGRT** — Oxford COVID-19 Government Response Tracker (policy data)

### Countries Analysed
- United States (USA)
- United Kingdom (GBR)
- Sweden (SWE)
- New Zealand (NZL)
- Germany (DEU)
- Turkey (TUR)
- Brazil (BRA)

---

## 1. Imports

In [ ]:
# Standard libraries
import sys
import warnings
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Matplotlib settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

# Add the src/ folder to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

## 2. Data Download

Downloading the raw datasets from JHU, OWID, and OxCGRT.

In [ ]:
from src.data_loader import download_all, get_file_path

download_results = download_all()

print("\nDownload Summary:")
for dataset, status in download_results.items():
    marker = "OK " if status else "FAIL"
    print(f"  [{marker}] {dataset}")

## 3. Data Preprocessing and Merging

In this stage we:
1. Convert the JHU data from wide to long format (`melt`)
2. Standardise country names to ISO-3 codes
3. Merge the three sources on `Country` and `Date`
4. Apply feature engineering (7-day rolling averages, per-million metrics)

In [ ]:
from src.preprocessor import prepare_master_dataset, load_master_dataset

df = prepare_master_dataset(save=True)

print(f"\nMaster dataset shape: {df.shape}")
print(f"Countries: {df['Country_ISO3'].unique().tolist()}")

In [ ]:
# First few rows
df.head(10)

In [ ]:
# Dataset summary
print("\nDataset Information:")
print(f"  Total rows : {len(df):,}")
print(f"  Columns    : {len(df.columns)}")
print(f"  Date range : {df['Date'].min().date()} - {df['Date'].max().date()}")
print(f"  Countries  : {df['Country_ISO3'].nunique()}")

print("\nKey statistics:")
df[['Cases', 'Deaths', 'New_Cases_7day_Avg', 'Stringency_Index']].describe().round(2)

In [ ]:
# Missing-value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing': missing,
    'Percent': missing_pct
}).sort_values('Percent', ascending=False)

print("\nMissing-value analysis (most missing columns):")
missing_df[missing_df['Missing'] > 0].head(15)

---

## 4. Visualisations

We generate five publication-quality figures:

1. **Policy Impact (dual axis)** — Stringency Index vs case counts
2. **Vaccination Efficacy** — vaccination rate vs deaths (scatter)
3. **Lag Analysis** — delayed effect of policy on cases
4. **Stringency Heatmap** — country-by-country policy intensity over time
5. **Event Study** — pre/post lockdown analysis

In [ ]:
from src.visualizer import (
    plot_policy_impact,
    plot_vaccination_efficacy,
    plot_lag_analysis,
    plot_stringency_heatmap,
    plot_event_study,
    plot_country_comparison
)

### 4.1 Policy Impact: Cases vs Stringency Index

This figure shows the relationship between government policy strictness (Stringency Index) and daily new cases.

**Reading:** when the Stringency Index rises (stricter measures), case counts tend to fall a few weeks later.

In [ ]:
# Policy impact for the United States
fig = plot_policy_impact(
    df,
    country='USA',
    date_start='2020-03-01',
    date_end='2022-12-31',
    show=True
)

In [ ]:
# Policy impact for Turkey
fig = plot_policy_impact(
    df,
    country='TUR',
    date_start='2020-03-01',
    date_end='2022-12-31',
    show=True
)

### 4.2 Vaccination Efficacy: Vaccination Rate vs Deaths

Scatter plot showing how mortality changes as the share of fully vaccinated population grows.

**Hypothesis:** higher vaccination coverage should reduce COVID-19 deaths.

In [ ]:
fig = plot_vaccination_efficacy(df, show=True)

### 4.3 Lag Analysis: Delayed Effect of Policy on Cases

This analysis quantifies how many days after a change in policy strictness the effect on case counts appears.

**Methodology:** correlate the Stringency Index with future case counts at varying lags.

In [ ]:
# Lag analysis for the United States
fig = plot_lag_analysis(df, country='USA', max_lag=30, show=True)

In [ ]:
# Lag analysis for Germany
fig = plot_lag_analysis(df, country='DEU', max_lag=30, show=True)

### 4.4 Stringency Heatmap: Cross-Country Policy Intensity

Heatmap comparing each country's policy strictness over time.

**Color scale:**
- Green = low Stringency (loose restrictions)
- Red   = high Stringency (strict restrictions)

In [ ]:
fig = plot_stringency_heatmap(
    df,
    date_start='2020-03-01',
    date_end='2022-06-30',
    show=True
)

### 4.5 Event Study: Pre/Post Lockdown Analysis

We compare case trajectories before and after a specific policy event.

**UK example:** 23 March 2020 — first national lockdown.

In [ ]:
# UK first national lockdown event study
fig = plot_event_study(
    df,
    country='GBR',
    event_date='2020-03-23',
    event_name='First National Lockdown',
    window_days=60,
    show=True
)

In [ ]:
# Turkey weekend curfew event study (April 2020)
fig = plot_event_study(
    df,
    country='TUR',
    event_date='2020-04-11',
    event_name='Weekend Curfew Implementation',
    window_days=45,
    show=True
)

### 4.6 Bonus: Cross-Country Comparison

Comparing per-million case counts across all countries.

In [ ]:
fig = plot_country_comparison(
    df,
    metric='New_Cases_7day_Per_Million',
    date_start='2020-03-01',
    date_end='2022-12-31',
    show=True
)

---

## 5. Findings and Conclusions

### Key findings

#### 1. Policy effectiveness
- A **negative correlation** is observed between the Stringency Index (government strictness) and case counts.
- Policy effects typically appear with a **7–14 day lag** (incubation period plus testing delays).

#### 2. Vaccination effect
- As the fully vaccinated share grows, deaths show a **clear downward trend**.
- The strongest effect appears once vaccination coverage exceeds ~50%.

#### 3. Cross-country differences
- **Sweden** pursued a looser strategy and saw higher early case counts.
- **New Zealand** kept COVID-19 under tight control for a long period via early, strict measures.
- **Turkey and Brazil** show oscillating curves driven by frequently changing policy.

### Limitations
- Data quality varies across countries.
- Changes in testing capacity affect case counts.
- Correlation is not causation.

---

In [ ]:
# Summary statistics by country
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

summary = df.groupby('Country').agg({
    'Cases': 'max',
    'Deaths': 'max',
    'Stringency_Index': 'mean',
    'people_fully_vaccinated_per_hundred': 'max'
}).round(1)

summary.columns = ['Total Cases', 'Total Deaths', 'Avg Stringency', 'Max Vaccination Rate (%)']
summary['Total Cases']  = summary['Total Cases'].apply(lambda x: f"{x:,.0f}")
summary['Total Deaths'] = summary['Total Deaths'].apply(lambda x: f"{x:,.0f}")

display(summary)

In [ ]:
print("\nAnalysis complete.")